# Multiple Choice Question

In [1]:
import pandas as pd

DATA_PATH = '../data/raw/'

Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [2]:
df_orders = pd.read_csv(DATA_PATH + 'orders.csv')

df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])

# 3. Sắp xếp theo khách hàng và thời gian
df_orders = df_orders.sort_values(by=['customer_id', 'order_date'])

# 4. Tính khoảng cách ngày giữa các đơn hàng liên tiếp của mỗi khách hàng
# Hàm diff() sẽ tính chênh lệch giữa dòng hiện tại và dòng trước đó
df_orders['inter_order_gap'] = df_orders.groupby('customer_id')['order_date'].diff().dt.days

# 5. Loại bỏ các giá trị NaN (đơn hàng đầu tiên của mỗi người) và tính trung vị
median_gap = df_orders['inter_order_gap'].dropna().median()

print(f"Trung vị số ngày giữa hai lần mua liên tiếp là: {median_gap} ngày")

Trung vị số ngày giữa hai lần mua liên tiếp là: 144.0 ngày


Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?

In [3]:
df_products = pd.read_csv(DATA_PATH + 'products.csv')

# 2. Tính tỷ suất lợi nhuận gộp (Gross Profit Margin) cho từng sản phẩm
# Công thức: (price - cogs) / price
df_products['margin'] = (df_products['price'] - df_products['cogs']) / df_products['price']

# 3. Tính trung bình tỷ suất lợi nhuận theo từng phân khúc (segment)
# Sau đó sắp xếp giảm dần để tìm phân khúc cao nhất
segment_performance = df_products.groupby('segment')['margin'].mean().sort_values(ascending=False)

# 4. Lấy tên phân khúc đứng đầu
highest_segment = segment_performance.idxmax()
highest_value = segment_performance.max()

print(f"Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất là: {highest_segment}")
print(f"Giá trị trung bình xấp xỉ: {highest_value:.4f} (tương đương {highest_value*100:.2f}%)")

Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất là: Standard
Giá trị trung bình xấp xỉ: 0.3134 (tương đương 31.34%)


Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [4]:
df_returns = pd.read_csv(DATA_PATH + 'returns.csv')
df_products = pd.read_csv(DATA_PATH + 'products.csv')

# 2. Thực hiện join (nối) hai bảng dựa trên cột 'product_id'
# Chúng ta sử dụng inner join để chỉ lấy những bản ghi trả hàng có thông tin sản phẩm tương ứng
merged_df = pd.merge(df_returns, df_products, on='product_id')

# 3. Lọc các bản ghi thuộc danh mục 'Streetwear'
streetwear_returns = merged_df[merged_df['category'] == 'Streetwear']

# 4. Đếm số lần xuất hiện của từng lý do trả hàng (return_reason)
# idxmax() sẽ trả về giá trị (lý do) có số lần xuất hiện cao nhất
reason_counts = streetwear_returns['return_reason'].value_counts()
top_reason = reason_counts.idxmax()
top_count = reason_counts.max()

print(f"Lý do trả hàng phổ biến nhất của danh mục Streetwear là: '{top_reason}'")
print(f"Số lượng bản ghi: {top_count}")

Lý do trả hàng phổ biến nhất của danh mục Streetwear là: 'wrong_size'
Số lượng bản ghi: 7626


Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [5]:
df_traffic = pd.read_csv(DATA_PATH + 'web_traffic.csv')

# 2. Nhóm theo traffic_source và tính trung bình bounce_rate
# Sau đó sắp xếp tăng dần (nguồn có tỷ lệ thấp nhất sẽ nằm ở đầu)
source_performance = df_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()

# 3. Lấy thông tin nguồn có tỷ lệ thoát thấp nhất
best_source = source_performance.idxmin()
best_value = source_performance.min()

print(f"Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất là: {best_source}")
print(f"Tỷ lệ thoát trung bình: {best_value:.4f} (hoặc {best_value*100:.2f}%)")

Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất là: email_campaign
Tỷ lệ thoát trung bình: 0.0045 (hoặc 0.45%)


Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

In [6]:
df_items = pd.read_csv(DATA_PATH + 'order_items.csv')

# 2. Tính tổng số dòng trong file
total_rows = len(df_items)

# 3. Đếm số dòng mà promo_id không phải là giá trị Null (NaN)
# Hàm .count() trong pandas mặc định chỉ đếm các giá trị không null
# promo_count = df_items['promo_id'].count()
promo_count = df_items[(df_items['promo_id'].notnull()) | (df_items['promo_id_2'].notnull())].shape[0]

# 4. Tính tỷ lệ phần trăm
promo_percentage = (promo_count / total_rows) * 100

print(f"Tổng số dòng: {total_rows}")
print(f"Số dòng có áp dụng khuyến mãi: {promo_count}")
print(f"Tỷ lệ phần trăm xấp xỉ: {promo_percentage:.2f}%")

Tổng số dòng: 714669
Số dòng có áp dụng khuyến mãi: 276316
Tỷ lệ phần trăm xấp xỉ: 38.66%


C:\Users\vnviv\AppData\Local\Temp\ipykernel_24684\1250578938.py:1: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items = pd.read_csv(DATA_PATH + 'order_items.csv')


Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [7]:
df_customers = pd.read_csv(DATA_PATH + 'customers.csv')
df_orders = pd.read_csv(DATA_PATH + 'orders.csv')

# 2. Loại bỏ các khách hàng không có thông tin nhóm tuổi (age_group is null)
df_customers = df_customers[df_customers['age_group'].notnull()]

# 3. Tính tổng số đơn hàng cho mỗi nhóm tuổi
# Đầu tiên, nối bảng orders với bảng customers để biết mỗi đơn hàng thuộc về nhóm tuổi nào
merged_df = pd.merge(df_orders, df_customers[['customer_id', 'age_group']], on='customer_id')
orders_per_group = merged_df.groupby('age_group')['order_id'].count()

# 4. Tính tổng số khách hàng trong mỗi nhóm tuổi
customers_per_group = df_customers.groupby('age_group')['customer_id'].nunique()

# 5. Tính trung bình số đơn hàng trên mỗi khách hàng
# Công thức: Tổng số đơn / Tổng số khách hàng (trong cùng một nhóm)
avg_orders_per_group = (orders_per_group / customers_per_group).sort_values(ascending=False)

# 6. Tìm nhóm cao nhất
highest_group = avg_orders_per_group.idxmax()
highest_val = avg_orders_per_group.max()

print(f"Nhóm tuổi có số đơn hàng trung bình cao nhất là: {highest_group}")
print(f"Giá trị trung bình: {highest_val:.2f} đơn hàng/khách hàng")

Nhóm tuổi có số đơn hàng trung bình cao nhất là: 55+
Giá trị trung bình: 5.41 đơn hàng/khách hàng


Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

In [8]:
df_geo = pd.read_csv(DATA_PATH + 'geography.csv')
df_orders = pd.read_csv(DATA_PATH + 'orders.csv')
df_items = pd.read_csv(DATA_PATH + 'order_items.csv')

# 2. Tính doanh thu cho từng dòng trong chi tiết đơn hàng
# Doanh thu = số lượng * đơn giá (sau khuyến mãi)
df_items['line_revenue'] = df_items['quantity'] * df_items['unit_price']

# 3. Kết hợp các bảng lại với nhau
# Nối items với orders để lấy mã zip của mỗi đơn hàng
order_details = pd.merge(df_items, df_orders[['order_id', 'zip']], on='order_id')

# Nối với geography để lấy thông tin vùng (region) dựa trên mã zip
full_data = pd.merge(order_details, df_geo[['zip', 'region']], on='zip')

# 4. Nhóm theo vùng và tính tổng doanh thu
regional_revenue = full_data.groupby('region')['line_revenue'].sum().sort_values(ascending=False)

# 5. Kết quả
top_region = regional_revenue.idxmax()
top_revenue = regional_revenue.max()

print(f"Vùng tạo ra tổng doanh thu cao nhất là: {top_region}")
print(f"Tổng doanh thu của vùng này là: {top_revenue:,.2f}")

# Hiển thị danh sách các vùng để so sánh
print("\nBảng xếp hạng doanh thu theo vùng:")
print(regional_revenue)

Vùng tạo ra tổng doanh thu cao nhất là: East
Tổng doanh thu của vùng này là: 7,637,532,676.20

Bảng xếp hạng doanh thu theo vùng:
region
East       7.637533e+09
Central    4.941908e+09
West       3.851035e+09
Name: line_revenue, dtype: float64


C:\Users\vnviv\AppData\Local\Temp\ipykernel_24684\1503249226.py:3: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items = pd.read_csv(DATA_PATH + 'order_items.csv')


Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

In [9]:
df_orders = pd.read_csv(DATA_PATH + 'orders.csv')

# 2. Lọc các đơn hàng có trạng thái là 'cancelled'
# Lưu ý: Kiểm tra chính xác chữ hoa/thường của giá trị trong file (ví dụ: 'Cancelled' hoặc 'cancelled')
cancelled_orders = df_orders[df_orders['order_status'].str.lower() == 'cancelled']

# 3. Đếm số lần xuất hiện của từng phương thức thanh toán (payment_method)
payment_counts = cancelled_orders['payment_method'].value_counts()

# 4. Lấy phương thức thanh toán phổ biến nhất
most_common_payment = payment_counts.idxmax()
count_value = payment_counts.max()

print(f"Phương thức thanh toán được sử dụng nhiều nhất cho các đơn hàng bị hủy là: {most_common_payment}")
print(f"Số lượng đơn hàng: {count_value}")

Phương thức thanh toán được sử dụng nhiều nhất cho các đơn hàng bị hủy là: credit_card
Số lượng đơn hàng: 28452


Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

In [10]:
df_returns = pd.read_csv(DATA_PATH + 'returns.csv')
df_items = pd.read_csv(DATA_PATH + 'order_items.csv')
df_products = pd.read_csv(DATA_PATH + 'products.csv')

# 2. Gắn thông tin 'size' vào bảng returns và order_items bằng cách join với products
# Chúng ta chỉ cần cột product_id và size từ bảng products
returns_with_size = pd.merge(df_returns, df_products[['product_id', 'size']], on='product_id')
items_with_size = pd.merge(df_items, df_products[['product_id', 'size']], on='product_id')

# 3. Đếm số bản ghi trả hàng cho mỗi kích thước
return_counts = returns_with_size.groupby('size').size()

# 4. Đếm tổng số dòng (lần mua) cho mỗi kích thước
item_counts = items_with_size.groupby('size').size()

# 5. Tính tỷ lệ trả hàng (Return Rate)
# Công thức: Số bản ghi returns / Số dòng trong order_items
# Sau đó lọc đúng 4 kích thước yêu cầu: S, M, L, XL
return_rates = (return_counts / item_counts).reindex(['S', 'M', 'L', 'XL'])

# 6. Tìm kích thước có tỷ lệ cao nhất
highest_size = return_rates.idxmax()
highest_rate = return_rates.max()

print(f"Kích thước có tỷ lệ trả hàng cao nhất là: {highest_size}")
print(f"Tỷ lệ trả hàng xấp xỉ: {highest_rate:.4f} ({highest_rate*100:.2f}%)")

# Hiển thị bảng so sánh giữa các kích thước
print("\nBảng so sánh tỷ lệ trả hàng theo kích thước:")
print(return_rates.sort_values(ascending=False))

Kích thước có tỷ lệ trả hàng cao nhất là: S
Tỷ lệ trả hàng xấp xỉ: 0.0565 (5.65%)

Bảng so sánh tỷ lệ trả hàng theo kích thước:
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64


C:\Users\vnviv\AppData\Local\Temp\ipykernel_24684\424588375.py:2: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items = pd.read_csv(DATA_PATH + 'order_items.csv')


Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

In [11]:
df_payments = pd.read_csv(DATA_PATH + 'payments.csv')

# 2. Nhóm theo số kỳ trả góp (installments) và tính giá trị thanh toán trung bình
# Sau đó sắp xếp giảm dần để tìm giá trị cao nhất
installment_analysis = df_payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)

# 3. Lấy thông tin kế hoạch có giá trị trung bình cao nhất
best_plan = installment_analysis.idxmax()
highest_avg_value = installment_analysis.max()

print(f"Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất là: {best_plan} kỳ")
print(f"Giá trị trung bình mỗi đơn hàng: {highest_avg_value:,.2f}")

# Hiển thị bảng chi tiết để so sánh giữa các kế hoạch
print("\nBảng thống kê giá trị trung bình theo số kỳ trả góp:")
print(installment_analysis)

Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất là: 6 kỳ
Giá trị trung bình mỗi đơn hàng: 24,446.65

Bảng thống kê giá trị trung bình theo số kỳ trả góp:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64
